# Lista 4 – Identyfikacja wzorców morfologicznych płodu (CTG)

**Autorzy:** Katarzyna Fojcik, Joanna Szołomicka, Teddy Ferdinan  
**Data:** 7 marca 2025

**Cel ćwiczenia:**
1. Poznanie prostych algorytmów uczenia maszynowego (Naiwny Klasyfikator Bayesa, Drzewo Decyzyjne, PCA)
2. Praktyczna realizacja kroków projektu ML: eksploracja danych, przygotowanie, dobór algorytmów i hiperparametrów, ocena i poprawa wyników

## 2. Wprowadzenie teoretyczne

W tej części zaimplementujemy podstawowe formuły:
- Entropia
- Przyrost wiedzy (Information Gain)
- Naiwny Klasyfikator Bayesa (wersja Gaussowska)
- PCA (Principal Component Analysis)
- Metryki: accuracy, precision, recall, F1-score

In [2]:
import numpy as np
import pandas as pd
from collections import defaultdict

### 2.4 Entropia i Przyrost wiedzy
- Entropia: $H(S) = -\sum_{x\in X} p(x) \log_2 p(x)$
- Przyrost wiedzy: $G(A) = H(S) - \sum_{v\in Values(A)} \frac{|S_v|}{|S|} H(S_v)$

In [3]:
def entropy(y: np.ndarray) -> float:
    """Oblicza entropię zbioru etykiet y."""
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-12))  # unikanie log(0)

In [4]:
def information_gain(X: np.ndarray, y: np.ndarray, feature_index: int) -> float:
    """Oblicza przyrost wiedzy dla cechy o indeksie feature_index (próg: mediana)."""
    H_S = entropy(y)
    # dzielimy według mediany
    threshold = np.median(X[:, feature_index])
    mask = X[:, feature_index] <= threshold
    y_left, y_right = y[mask], y[~mask]
    H_left, H_right = entropy(y_left), entropy(y_right)
    w_left = len(y_left) / len(y)
    w_right = len(y_right) / len(y)
    return H_S - (w_left * H_left + w_right * H_right)

### 2.5 Naiwny Klasyfikator Bayesa (Gauss)
Model zakłada dla każdej cechy Xi i klasy y: Xi|y ~ N(mu_{i,y}, sigma_{i,y}^2)

In [5]:
class GaussianNB_scratch:
    def __init__(self):
        self.class_prior_ = {}
        self.means_ = {}
        self.vars_ = {}

    def fit(self, X: np.ndarray, y: np.ndarray):
        classes = np.unique(y)
        n = len(y)
        for c in classes:
            X_c = X[y == c]
            self.class_prior_[c] = len(X_c) / n
            self.means_[c] = X_c.mean(axis=0)
            self.vars_[c] = X_c.var(axis=0) + 1e-9  # stabilizacja

    def _gaussian_probability(self, x, mean, var):
        coeff = 1.0 / np.sqrt(2.0 * np.pi * var)
        exponent = np.exp(-((x - mean) ** 2) / (2 * var))
        return coeff * exponent

    def predict(self, X: np.ndarray) -> np.ndarray:
        n_samples, n_features = X.shape
        posteriors = []
        for x in X:
            class_probs = {}
            for c, prior in self.class_prior_.items():
                class_probs[c] = np.log(prior)
                mean, var = self.means_[c], self.vars_[c]
                # dodajemy log-prawdopodobieństwa cech
                class_probs[c] += np.sum(np.log(self._gaussian_probability(x, mean, var)))
            posteriors.append(max(class_probs, key=class_probs.get))
        return np.array(posteriors)

### 2.6 PCA (Principal Component Analysis)
Transformacja danych w celu zmniejszenia wymiaru przy zachowaniu największej wariancji.

In [8]:
from sklearn.decomposition import PCA

def perform_pca(X: np.ndarray, n_components: int):
    """Zwraca dane przekształcone PCA oraz obiekt PCA."""
    pca = PCA(n_components=n_components)
    X_p = pca.fit_transform(X)
    return X_p, pca

### 2.7 Metryki oceny klasyfikacji
- Accuracy (dokładność)
- Precision (precyzja)
- Recall (czułość)
- F1-score

In [9]:
def accuracy_score_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(y_true == y_pred)

def precision_score_scratch(y_true: np.ndarray, y_pred: np.ndarray, positive_label) -> float:
    tp = np.sum((y_pred == positive_label) & (y_true == positive_label))
    fp = np.sum((y_pred == positive_label) & (y_true != positive_label))
    return tp / (tp + fp + 1e-12)

def recall_score_scratch(y_true: np.ndarray, y_pred: np.ndarray, positive_label) -> float:
    tp = np.sum((y_pred == positive_label) & (y_true == positive_label))
    fn = np.sum((y_pred != positive_label) & (y_true == positive_label))
    return tp / (tp + fn + 1e-12)

def f1_score_scratch(y_true: np.ndarray, y_pred: np.ndarray, positive_label) -> float:
    p = precision_score_scratch(y_true, y_pred, positive_label)
    r = recall_score_scratch(y_true, y_pred, positive_label)
    return 2 * p * r / (p + r + 1e-12)

## 3. Zadanie – Identyfikacja wzorców morfologicznych płodu

In [10]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA as SKPCA
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

### 3.1 Eksploracja danych
1. Wczytanie `cardiotocography_v2.csv`
2. Podstawowe statystyki i analiza braków danych

In [11]:
df = pd.read_csv('cardiotocography_v2.csv')
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'cardiotocography_v2.csv'

In [12]:
# Statystyki opisowe i brakujące wartości
df.describe().T

df.isnull().sum()

NameError: name 'df' is not defined

### 3.2 Przygotowanie danych
- Podział na zbiór treningowy i walidacyjny
- Radzenie sobie z brakującymi wartościami (SimpleImputer)
- Preprocessing: normalizacja (MinMax), standaryzacja
- PCA (opcjonalnie)

In [ ]:
# Podział danych
y = df['CLASS'].values
nX = df.drop('CLASS', axis=1).values
X_train, X_val, y_train, y_val = train_test_split(nX, y, test_size=0.3, random_state=42, stratify=y)

# Imputacja braków
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)

# Normalizacja i standaryzacja
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train_imp)
X_val_mm = scaler_mm.transform(X_val_imp)

scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train_imp)
X_val_std = scaler_std.transform(X_val_imp)

# PCA z 5 składowymi
dim = 5
X_train_pca, pca_model = perform_pca(X_train_std, dim)
X_val_pca = pca_model.transform(X_val_std)

### 3.3 Klasyfikacja
Testujemy:
- GaussianNB (z domyślnymi i zmienionym var_smoothing)
- DecisionTreeClassifier (różne max_depth)

In [13]:
# GaussianNB
gnb_default = GaussianNB()
gnb_default.fit(X_train_imp, y_train)

y_pred_nb = gnb_default.predict(X_val_imp)
print("GaussianNB (default) Report:")
print(classification_report(y_val, y_pred_nb))

# Decision Tree z różną głębokością
depths = [3, 5, None]
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_imp, y_train)
    y_pred_dt = dt.predict(X_val_imp)
    print(f"DecisionTree (max_depth={d}):")
    print(classification_report(y_val, y_pred_dt))

NameError: name 'X_train_imp' is not defined

### Bonus: Random Forest i SVM oraz eksperyment z regularyzacją (przeuczenie)
(Dodatkowa implementacja, jeśli czas pozwoli)

## 4. Ocena i interpretacja wyników
W tej sekcji zbierzemy wyniki w tabeli i porównamy metryki dla różnych metod.

In [ ]:
results = pd.DataFrame()

In [ ]:
results